#IMPORT

In [2]:
!pip install google-play-scraper
!pip install langdetect tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=0b66bd133ca3567bff8d84c3ea2f06499152ac9af1556d968af06ec040d85575
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [3]:
from google_play_scraper import reviews, Sort
import pandas as pd
import csv
from datetime import datetime
from tqdm import tqdm
import time
from langdetect import detect, DetectorFactory

#KONFIGURASI

In [ ]:
# Supaya hasil deteksi stabil
DetectorFactory.seed = 0

APP_ID = "com.kurogame.wutheringwaves.global"
LOCALE = "en"             # target bahasa review (English)
COUNTRY = "us"            # negara review
MAX_REVIEWS = 20000        # jumlah maksimal review
BATCH_SIZE = 200          # jumlah review per request
SLEEP_BETWEEN_BATCH = 1.0 # jeda antar request (detik)
OUTPUT_CSV = f"reviews_{APP_ID}_EN_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

#SCRAPPING

In [9]:
def fetch_reviews(app_id, max_reviews=1000, batch_size=200, sleep_between=1.0, locale="en", country="us"):
    all_reviews = []
    continuation_token = None
    pbar = tqdm(total=max_reviews, desc="Mengambil review")

    while len(all_reviews) < max_reviews:
        to_fetch = min(batch_size, max_reviews - len(all_reviews))
        result, continuation_token = reviews(
            app_id,
            lang=locale,
            country=country,
            sort=Sort.NEWEST,   # bisa juga Sort.MOST_RELEVANT
            count=to_fetch,
            continuation_token=continuation_token
        )

        if not result:
            break

        all_reviews.extend(result)
        pbar.update(len(result))

        if not continuation_token:
            break

        time.sleep(sleep_between)

    pbar.close()
    return all_reviews[:max_reviews]

#FILTER BAHASA

In [10]:
def filter_english(reviews_list):
    english_reviews = []
    for r in reviews_list:
        try:
            if r.get("content"):
                lang = detect(r["content"])
                if lang == "en":
                    english_reviews.append(r)
        except:
            # kalau deteksi gagal, skip
            continue
    return english_reviews

#MAIN

In [11]:
if __name__ == "__main__":
    print(f"Scraping hingga {MAX_REVIEWS} review untuk app: {APP_ID}")
    raw_data = fetch_reviews(APP_ID, max_reviews=MAX_REVIEWS, batch_size=BATCH_SIZE,
                             sleep_between=SLEEP_BETWEEN_BATCH, locale=LOCALE, country=COUNTRY)

    print(f"Jumlah review sebelum filter: {len(raw_data)}")
    english_data = filter_english(raw_data)
    print(f"Jumlah review berbahasa Inggris: {len(english_data)}")

    df = pd.DataFrame(english_data)
    df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_NONNUMERIC)
    print(f"Berhasil menyimpan {len(df)} review ke file: {OUTPUT_CSV}")

Scraping hingga 20000 review untuk app: com.kurogame.wutheringwaves.global


Mengambil review: 100%|██████████| 20000/20000 [02:04<00:00, 160.76it/s]


Jumlah review sebelum filter: 20000
Jumlah review berbahasa Inggris: 15210
Berhasil menyimpan 15210 review ke file: reviews_com.kurogame.wutheringwaves.global_EN_20251003_022130.csv
